# SPHEREx Self-calibration Pipeline

In [ ]:
import glob
import os
import h5py
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 400 # User can set this outside the class if needed

from astropy.io import fits
from astropy.wcs import WCS

import importlib
sys.path.insert(0, '/home/thomasli/spherex/selfcal')
from py import MakeMap

import os
import h5py
import sys 
from tqdm import tqdm
from multiprocessing import Pool 
from concurrent.futures import ProcessPoolExecutor, as_completed, ThreadPoolExecutor
import numpy as np

from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
import astropy.units as u
from astropy.table import Table

from reproject import reproject_interp, reproject_exact, reproject_adaptive

from scipy.sparse import coo_matrix
from scipy.sparse.linalg import lsqr
from functools import partial

from py.WCSHelper import load_from_fits, save_to_fits, find_optimal_frame

from IPython.display import clear_output

NameError: name 'sys' is not defined

# Wrapper for Selfcal Pipeline

In [ ]:
class Reprojector:
    def __init__(self, config, exposure_list=None):
        '''Initialize path to reference WCS and reprojected files'''
        self.config = config
        # check if the config has the required keys
        required_keys = ['output_dir', 'run_name', 'resolution_arcsec']
        for key in required_keys:
            if key not in self.config:
                raise ValueError(f"Configuration must include '{key}'")

        self.exposure_list = exposure_list
        if 'ref_path' not in self.config:
            self.config['ref_path'] = os.path.join(self.config['output_dir'], self.config['run_name'], 'ref.fits')
        if 'reproj_dir' not in self.config:
            self.config['reproj_dir'] = os.path.join(self.config['output_dir'], self.config['run_name'], 'reprojected')
        if not os.path.exists(self.config['reproj_dir']):
            os.makedirs(self.config['reproj_dir'])

        self.ref_shape = None
        self.ref_wcs = None

    def define_reference(self, padding_pixels=100, use_ext=[1]):
        '''Define the smallest WCS oriented north-up, east-left frame that can contain all exposures'''
        if not os.path.exists(self.config['ref_path']):
            print(f"Reference WCS not found at {self.config['ref_path']}. Creating a new reference frame.")
            self.ref_wcs, self.ref_shape = find_optimal_frame(
                exposure_list=self.exposure_list,
                resolution_arcsec=self.config['resolution_arcsec'],
                padding_pixels=padding_pixels,
                use_ext=use_ext
            )
            save_to_fits(self.ref_wcs, self.ref_shape, os.path.join(self.config['output_dir'], self.config['run_name'], 'ref.fits'))
            print(f"Reference WCS saved to {self.config['ref_path']}")
        else:
            self.ref_wcs, self.ref_shape = load_from_fits(self.config['ref_path'])
        print(f'Mosaic shape: {self.ref_shape}')
        print(f'Mosaic WCS: {self.ref_wcs}')

    def run_reproject(self, max_workers=50, method='exact', padding_percentage=0.05, oversample_factor=2, 
                      sci_ext_list=None, dq_ext_list=None, exp_idx_list=None, det_idx_list=None,
                      output_dir=None, replace_existing=False):
        if self.ref_wcs is None or self.ref_shape is None:
            raise ValueError("Reference WCS and shape must be defined before running reprojection. Call define_reference() first.")
        if output_dir is None:
            output_dir = self.config['reproj_dir']
        self.reproj_list = MakeMap.batch_reproject(
            # Can edit
            num_processes = max_workers, 
            method = method,  # interp: fastest, adaptive: conserves flux, exact: most accurate

            # Porbably don't want to edit
            exposure_list = self.exposure_list,
            ref_wcs = self.ref_wcs, 
            ref_shape = self.ref_shape,
            output_dir = output_dir, 
            padding_percentage = padding_percentage,
            oversample_factor = oversample_factor,
            sci_ext_list = sci_ext_list, 
            dq_ext_list = dq_ext_list,
            exp_idx_list = exp_idx_list,
            det_idx_list = det_idx_list,
            replace_existing = replace_existing
            )
        
    def check_reproj_files(self):
        for f in tqdm(self.reproj_list):
            result = MakeMap.load_reproj_file(f, fields=['sub_data',])
            if result['_is_missing_']:
                os.remove(f)
                print(f"Removed {f} due to missing data")

    def get_reproj_files(self, reproj_dir=None):
        if reproj_dir is None:
            reproj_dir = self.config['reproj_dir']
        self.reproj_list = sorted(glob.glob(os.path.join(reproj_dir, '*.h5')))
        self.det_idx_list = []
        self.exp_idx_list = []
        for file in tqdm(self.reproj_list):
            file_name = os.path.basename(file)
            self.det_idx_list.append(int(file_name[file_name.find('det_')+4:file_name.find('det_')+6]))
            self.exp_idx_list.append(int(file_name[file_name.find('exp_')+4:file_name.find('exp_')+8]))
        
class Calibrator(Reprojector):
    def __init__(self, config, reproj_dir=None):
        super().__init__(config)
        self.config['cal_dir'] = os.path.join(self.config['output_dir'], self.config['run_name'], 'calibration')
        self.get_reproj_files(reproj_dir)
        self.ref_wcs, self.ref_shape = load_from_fits(self.config['ref_path'])
        self.A = None
        self.B = None

    def setup_lsqr(self, apply_mask=True, apply_weight=True, chunk_map=None, chunk_valid_mask=None, max_workers=20, outlier_thresh=3.0):
        self.A, self.b = MakeMap.setup_lsqr(self.reproj_list, self.ref_shape, self.exp_idx_list, self.det_idx_list,
               apply_mask=apply_mask, apply_weight=apply_weight, chunk_map=chunk_map, chunk_valid_mask=chunk_valid_mask,
               max_workers=max_workers, outlier_thresh=outlier_thresh)
        
    def apply_lsqr(self, x0=None, atol=1e-06, btol=1e-06, damp=1e-2, iter_lim=300):
        if self.A is None or self.b is None:
            raise ValueError("LSQR matrix A and vector b must be set up before applying LSQR.")
        self.O, self.S, self.D = MakeMap.apply_lsqr(self.A, self.b, self.ref_shape, self.exp_idx_list, self.det_idx_list, 
                                                    x0=x0, atol=atol, btol=btol, damp=damp, iter_lim=iter_lim)
    
    def save_calibration(self, cal_dir=None, cal_file='cal.h5'):
        if cal_dir is None:
            cal_dir = self.config['cal_dir']
        if not os.path.exists(cal_dir):
            os.makedirs(cal_dir)

        cal_path = os.path.join(cal_dir, cal_file)
        with h5py.File(cal_path, 'w') as f:
            f.create_dataset('O', data=self.O, compression='gzip')
            f.create_dataset('S', data=self.S, compression='gzip')
            f.create_dataset('D', data=self.D, compression='gzip')
            f.create_dataset('reproj_list', data=np.array(self.reproj_list, dtype='S'))
        print(f"Calibration saved to {cal_path}")
        return cal_path

class Mosaicker(Reprojector):
    def __init__(self, config, reproj_dir=None):
        super().__init__(config)
        self.get_reproj_files(reproj_dir)
        self.ref_wcs, self.ref_shape = load_from_fits(self.config['ref_path'])
        self.config['mos_dir'] = os.path.join(self.config['output_dir'], self.config['run_name'], 'mosaic')
        self.cal_path = None
        self.O = None
        self.S = None
        self.D = None

    def load_calibration(self, cal_path):
        with h5py.File(cal_path, 'r') as f:
            self.O = f['O'][:]
            self.S = f['S'][:]
            self.D = f['D'][:]
        print(f"Calibration loaded from {cal_path}")
        self.cal_path = cal_path

    def make_mosaic(self, apply_mask=True, apply_weight=True, chunk_map=None, chunk_valid_mask=None, max_workers=20, 
    make_std_map=False, apply_sigma_clipping=False, sigma=2.0):
        
        if self.O is None or self.D is None:
            print("Waning: Calibration not loaded. No calibration will be applied to the mosaic.")

        mean, weight = MakeMap.compute_mean_map(
            ref_shape=self.ref_shape,
            reproj_file_list=self.reproj_list,
            exp_offset=self.O-np.mean(self.O, axis=0) if self.O is not None else None,
            det_offset=[self.D-np.mean(self.D[chunk_valid_mask==1], axis=0)] if self.D is not None else None,
            det_idx_list=self.det_idx_list,
            exp_idx_list=self.exp_idx_list,
            apply_weight=apply_weight,
            apply_mask=apply_mask,
            chunk_map=chunk_map,
            max_workers=max_workers,
            chunk_valid_mask = chunk_valid_mask,
        )
        if make_std_map:
            std, _ = MakeMap.compute_std_map(
                mean_map=mean,
                ref_shape=self.ref_shape,
                reproj_file_list=self.reproj_list,
                exp_offset=self.O-np.mean(self.O, axis=0) if self.O is not None else None,
                det_offset=[self.D-np.mean(self.D[chunk_valid_mask==1], axis=0)] if self.D is not None else None,
                det_idx_list=self.det_idx_list,
                exp_idx_list=self.exp_idx_list,
                apply_weight=apply_weight,
                apply_mask=apply_mask,
                chunk_map=chunk_map,
                max_workers=max_workers,
                chunk_valid_mask = chunk_valid_mask
            )
        if make_std_map and apply_sigma_clipping:
            sc_mean, weight = MakeMap.compute_sc_mean(
                mean_map=mean,
                std_map=std,
                sigma=sigma,
                ref_shape=self.ref_shape,
                reproj_file_list=self.reproj_list,
                exp_offset=self.O-np.mean(self.O, axis=0) if self.O is not None else None,
                det_offset=[self.D-np.mean(self.D[chunk_valid_mask==1], axis=0)] if self.D is not None else None,
                exp_idx_list=self.exp_idx_list,
                det_idx_list=self.det_idx_list,
                apply_weight=apply_weight,
                apply_mask=True,
                chunk_map=chunk_map,
                chunk_valid_mask=chunk_valid_mask,
                max_workers=max_workers
            )
        self.mean_map = mean
        self.std_map = std if make_std_map else None
        self.sc_mean = sc_mean if make_std_map and apply_sigma_clipping else None

        return self.mean_map, self.std_map, self.sc_mean

        
    def save_mosaic(self, mos_dir=None, mos_file='mosaic.fits', overwrite=False):
        if mos_dir is None:
            mos_dir = self.config['mos_dir']
        if not os.path.exists(mos_dir):
            os.makedirs(mos_dir)

        mos_path = os.path.join(mos_dir, mos_file)

        mosaic_hdu = fits.PrimaryHDU(data=self.sc_mean, header=self.ref_wcs.to_header())
        mosaic_hdu.header['NAXIS1'] = self.ref_shape[1]
        mosaic_hdu.header['NAXIS2'] = self.ref_shape[0]
        mosaic_hdu.header['NAXIS'] = 2
        mosaic_hdu.header['BUNIT'] = 'MJy/sr'
        mosaic_hdu.header['EXTNAME'] = 'MOSAIC'
        mosaic_hdu.header['CAL_FILE'] = self.cal_path

        std_hdu = fits.ImageHDU(data=self.std_map, header=self.ref_wcs.to_header())
        std_hdu.header['NAXIS1'] = self.ref_shape[1]
        std_hdu.header['NAXIS2'] = self.ref_shape[0]
        std_hdu.header['NAXIS'] = 2
        std_hdu.header['BUNIT'] = 'MJy/sr'
        std_hdu.header['EXTNAME'] = 'STD'

        hdul = fits.HDUList([mosaic_hdu, std_hdu])
        hdul.writeto(mos_path, overwrite=overwrite)
        print(f"Mosaic saved to {mos_path}")
        return mos_path


In [12]:
from skimage import measure
from scipy.interpolate import make_smoothing_spline

def interpolate_array(data_arr, interp_factor=5):
    interp_arr = np.hstack([
        np.linspace(data_arr[i], data_arr[i + 1], interp_factor, endpoint=False) 
        for i in range(len(data_arr) - 1)
    ] + [data_arr[-1]])  # Append the last element
    return interp_arr

def make_chunk_map(band, interp_factor=5,
                   calibration_file='/data1/SPHEREx/Data/Survey_3/calibs/SSDC_SpecCal_v2025Feb/SSDC_CENTER_all_v20250202.fits'):
    hdul = fits.open(calibration_file)
    wav_map = hdul[0].data[band-1]
    tbl = Table.read('/home/thomasli/spherex/spherex_nep_catalogues/spherex_channels.csv')
    sub_tbl = tbl[tbl['band'] == band]
    channel_edges = np.hstack([sub_tbl['lmin'].data, sub_tbl['lmax'].data[-1:]])
    fine_edges = interpolate_array(channel_edges, interp_factor=interp_factor)
    channel_idx = np.searchsorted(fine_edges, wav_map, side='right')

    mask_smooth = np.zeros_like(channel_idx)

    for i in tqdm(np.unique(channel_idx)[:]):
        mask = channel_idx >= i
        contours = measure.find_contours(mask, level=0.1, positive_orientation='low')
        if len(contours) == 0:
            continue
        contour = []
        for c in contours:
            if len(c) > 500:  # Filter out small contours
                contour.append(c)
        contour = np.concatenate(contour)
        ys, xs = contour[:, 0], contour[:, 1]

        # sort
        sorted_indices = np.argsort(xs)
        xs = xs[sorted_indices]
        ys = ys[sorted_indices]
        # remove duplicates
        unique_indices = np.unique(xs, return_index=True)[1]
        xs = xs[unique_indices]
        ys = ys[unique_indices]

        spl = make_smoothing_spline(xs, ys, lam=1e7)
        x_idx = np.arange(mask_smooth.shape[1])
        y_lim = spl(x_idx)
        y_lim = np.clip(y_lim, 0, mask_smooth.shape[0]-1)

        for x, y in zip(x_idx, y_lim):
            mask_smooth[:int(y+0.5), int(x)] = i

    return mask_smooth

def make_chunk_mask(valid_channels, interp_factor=5):
    chunk_valid_mask = np.zeros(17*interp_factor + 2)
    chunk_valid_mask[np.hstack(((np.array(valid_channels)-1)*interp_factor)[:, None] + np.arange(interp_factor)) + 1] = 1
    return chunk_valid_mask

def visualize_chunk_map(chunk_map, chunk_valid_mask):
    masked_chunk_map = np.where(chunk_valid_mask[chunk_map], chunk_map, np.nan)
    plt.imshow(masked_chunk_map, cmap='viridis', interpolation='none')

### Define Common Reference Frame

In [ ]:
exposure_list = glob.glob(f'/data1/SPHEREx/deep_north/*/*/*/*.fits')
for exp_file in exposure_list:
    hdul = fits.open(exp_file)
    header = hdul[1].header
    good_astrometry = header.get('FINAST', 2)
    if good_astrometry != 0:
        print(f"Skipping {exp_file} due to poor astrometry (FINAST={good_astrometry})")
        exposure_list.remove(exp_file)
print(f"Found {len(exposure_list)} exposures")


Skipping /data1/SPHEREx/deep_north/2025W20_2D/l2b-v3-2025-139/6/level2_2025W20_2D_0567_1D6_spx_l2b-v3-2025-139.fits due to poor astrometry (FINAST=3)
Skipping /data1/SPHEREx/deep_north/2025W20_2D/l2b-v3-2025-139/3/level2_2025W20_2D_0567_1D3_spx_l2b-v3-2025-139.fits due to poor astrometry (FINAST=1)
Skipping /data1/SPHEREx/deep_north/2025W20_2D/l2b-v3-2025-139/5/level2_2025W20_2D_0567_1D5_spx_l2b-v3-2025-139.fits due to poor astrometry (FINAST=2)
Skipping /data1/SPHEREx/deep_north/2025W20_2D/l2b-v3-2025-138/6/level2_2025W20_2D_0253_2D6_spx_l2b-v3-2025-138.fits due to poor astrometry (FINAST=1)
Skipping /data1/SPHEREx/deep_north/2025W20_2D/l2b-v3-2025-138/6/level2_2025W20_2D_0311_4D6_spx_l2b-v3-2025-138.fits due to poor astrometry (FINAST=2)
Skipping /data1/SPHEREx/deep_north/2025W20_2D/l2b-v3-2025-138/6/level2_2025W20_2D_0482_4D6_spx_l2b-v3-2025-138.fits due to poor astrometry (FINAST=2)
Skipping /data1/SPHEREx/deep_north/2025W20_2D/l2b-v3-2025-138/6/level2_2025W20_2D_0140_1D6_spx_l2b-v

In [21]:
ref_wcs, ref_shape = find_optimal_frame(
                exposure_list=exposure_list,
                resolution_arcsec=6.2,
                padding_pixels=100,
                use_ext=[1])

Defining optimal celestial WCS...


Loading corner WCS: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 10739/10739 [01:48<00:00, 99.31it/s]


In [ ]:
WCSHelper.save_to_fits(ref_wcs, ref_shape, '/home/thomasli/spherex/selfcal/outputs/common_ref.fits')

Reference frame FITS saved to: /home/thomasli/spherex/selfcal/outputs/common_ref.fits


In [7]:
detector = 2
exposure_list = glob.glob(f'/data1/SPHEREx/deep_north/*/*/*/*D{detector}*.fits')
for exp_file in exposure_list:
    hdul = fits.open(exp_file)
    header = hdul[1].header
    good_astrometry = header.get('FINAST', 2)
    if good_astrometry != 0:
        print(f"Skipping {exp_file} due to poor astrometry (FINAST={good_astrometry})")
        exposure_list.remove(exp_file)
print(f"Found {len(exposure_list)} exposures")


Skipping /data1/SPHEREx/deep_north/2025W17_4B/l2b-v1-2025-116/2/level2_2025W17_4B_0030_4D2_spx_l2b-v1-2025-116.fits due to poor astrometry (FINAST=1)
Skipping /data1/SPHEREx/deep_north/2025W17_4B/l2b-v1-2025-116/2/level2_2025W17_4B_0021_3D2_spx_l2b-v1-2025-116.fits due to poor astrometry (FINAST=1)
Skipping /data1/SPHEREx/deep_north/2025W20_1C/l2b-v3-2025-134/2/level2_2025W20_1C_0209_1D2_spx_l2b-v3-2025-134.fits due to poor astrometry (FINAST=2)
Found 1834 exposures


### Single Run Test

In [8]:
config = {}
config['output_dir'] = '/home/thomasli/spherex/selfcal/outputs'
config['run_name'] = f'nep_det{detector}_6p2arcsec'
config['resolution_arcsec'] = 6.2
# config['ref_path'] = '/home/thomasli/spherex/selfcal/outputs/common_ref.fits'

rr = Reprojector(config, exposure_list=exposure_list)

In [9]:
rr.define_reference(padding_pixels=100, use_ext=[1])

Loading reference frame from: /home/thomasli/spherex/selfcal/outputs/nep_det2_6p2arcsec/ref.fits
Mosaic shape: (7465, 8563)
Mosaic WCS: WCS Keywords

Number of WCS axes: 2
CTYPE : 'RA---TAN' 'DEC--TAN' 
CRVAL : 268.96647330007 65.780828426404 
CRPIX : 4475.5528907477 3807.4037035386 
PC1_1 PC1_2  : 1.0 0.0 
PC2_1 PC2_2  : 0.0 1.0 
CDELT : -0.0017222222222222 0.0017222222222222 
NAXIS : 8563  7465


In [36]:
rr.run_reproject(max_workers=60, method='exact', padding_percentage=0.05, oversample_factor=2, 
                      sci_ext_list=[1], 
                      dq_ext_list=[2], 
                      exp_idx_list=np.arange(0, len(exposure_list)), 
                      det_idx_list=[0]*len(exposure_list),
                      replace_existing=False)

Starting batch reprojection. Output will be saved to: /home/thomasli/spherex/selfcal/outputs/nep_det2_6p2arcsec/reprojected


Reprojecting frames: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 1834/1834 [00:01<00:00, 1360.58it/s]


Batch reprojection completed. 1834 frames successfully processed out of 1834.


In [51]:
rr.check_reproj_files()

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1837/1837 [06:16<00:00,  4.87it/s]


In [13]:
chunk_map = make_chunk_map(2, interp_factor=10)
chunk_valid_mask = make_chunk_mask([3, 4], interp_factor=10)


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 172/172 [00:10<00:00, 16.18it/s]


In [38]:
cc = Calibrator(config)

cc.setup_lsqr(
    apply_mask=True, 
    apply_weight=True, 
    chunk_map=chunk_map, 
    chunk_valid_mask=chunk_valid_mask, 
    max_workers=20, 
    outlier_thresh=3.0
    )

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1834/1834 [00:00<00:00, 1333163.52it/s]

Loading reference frame from: /home/thomasli/spherex/selfcal/outputs/nep_det2_6p2arcsec/ref.fits



Building A, b matrix: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 1834/1834 [07:22<00:00,  4.15it/s]


In [ ]:
cc.apply_lsqr(x0=None, atol=1e-05, btol=1e-05, damp=1e-2, iter_lim=200)

Solving least squares for 63924801 unknowns with 753806052 equations.
 
LSQR            Least-squares solution of  Ax = b
The matrix A has 753806052 rows and 63924801 columns
damp = 1.00000000000000e-02   calc_var =        0
atol = 1.00e-06                 conlim = 1.00e+08
btol = 1.00e-06               iter_lim =      300
 
   Itn      x[0]       r1norm     r2norm   Compatible    LS      Norm A   Cond A
     0  0.00000e+00   4.879e+04  4.879e+04    1.0e+00  1.6e-01
     1  0.00000e+00   7.444e+03  7.444e+03    1.5e-01  7.3e-01   7.8e+03  1.0e+00
     2  0.00000e+00   4.677e+03  4.677e+03    9.6e-02  2.1e-01   1.1e+04  2.0e+00
     3  0.00000e+00   4.316e+03  4.316e+03    8.8e-02  1.0e-01   1.2e+04  3.2e+00
     4  0.00000e+00   4.203e+03  4.203e+03    8.6e-02  5.1e-02   1.4e+04  4.6e+00
     5  0.00000e+00   4.166e+03  4.166e+03    8.5e-02  3.0e-02   1.5e+04  6.0e+00
     6  0.00000e+00   4.148e+03  4.148e+03    8.5e-02  2.0e-02   1.6e+04  7.7e+00
     7  0.00000e+00   4.141e+03  4.14

In [49]:
# import types

# cc.save_calibration = types.MethodType(Calibrator.save_calibration, cc)

In [50]:
cal_path = cc.save_calibration(cal_file='single_test.h5')

Calibration saved to /home/thomasli/spherex/selfcal/outputs/nep_det2_6p2arcsec/calibration/single_test.h5


In [105]:
mm = Mosaicker(config)
mm.load_calibration(cal_path=cal_path)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1834/1834 [00:00<00:00, 1351195.07it/s]


Loading reference frame from: /home/thomasli/spherex/selfcal/outputs/nep_det2_6p2arcsec/ref.fits
Calibration loaded from /home/thomasli/spherex/selfcal/outputs/nep_det2_6p2arcsec/calibration/single_test.h5


In [107]:
mean, std, sc = mm.make_mosaic(
    apply_mask=True, 
    apply_weight=True, 
    chunk_map=chunk_map, 
    chunk_valid_mask=chunk_valid_mask, 
    max_workers=40,
    make_std_map=True, 
    apply_sigma_clipping=True, 
    sigma=2.0
)

Sigma-clipped coadd: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [04:18<00:00,  6.45s/it]


In [120]:
#update mm method
mm.save_mosaic = types.MethodType(Mosaicker.save_mosaic, mm)

In [121]:
mm.save_mosaic(mos_file='single_test.fits', overwrite=True)

Mosaic saved to /home/thomasli/spherex/selfcal/outputs/nep_det2_6p2arcsec/mosaic/single_test.fits


'/home/thomasli/spherex/selfcal/outputs/nep_det2_6p2arcsec/mosaic/single_test.fits'

In [14]:
from IPython.display import clear_output

In [26]:
chunk_map

array([[86, 86, 86, ..., 86, 86, 86],
       [86, 86, 86, ..., 86, 86, 86],
       [86, 86, 86, ..., 86, 86, 86],
       ...,
       [ 1,  1,  1, ...,  1,  1,  1],
       [ 1,  1,  1, ...,  1,  1,  1],
       [ 0,  0,  0, ...,  0,  0,  0]], shape=(2040, 2040))

In [ ]:
chunk_map = make_chunk_map(2, interp_factor=5)
chs = [[1], [2], [3], [5], [6], [7], [8], [9], [10], [11], [12], [13], [14], [15], [16], [17]]

for ch in chs:
    ch_name = '-'.join([str(i) for i in ch])
    print(f"Processing channels {ch_name}")
    
    chunk_valid_mask = make_chunk_mask(ch, interp_factor=5)

    cc = Calibrator(config)

    cc.setup_lsqr(
        apply_mask=True, 
        apply_weight=True, 
        chunk_map=chunk_map, 
        chunk_valid_mask=chunk_valid_mask, 
        max_workers=20, 
        outlier_thresh=1.0
        )

    cc.apply_lsqr(x0=None, atol=1e-06, btol=1e-06, damp=1e-2, iter_lim=300)

    cal_path = cc.save_calibration(cal_file=f'cal_det2_ch{ch_name}.h5')

    mm = Mosaicker(config)
    mm.load_calibration(cal_path=cal_path)

    mean, std, sc = mm.make_mosaic(
        apply_mask=True, 
        apply_weight=True, 
        chunk_map=chunk_map, 
        chunk_valid_mask=chunk_valid_mask, 
        max_workers=40,
        make_std_map=True, 
        apply_sigma_clipping=True, 
        sigma=2.0
    )

    mm.save_mosaic(mos_file=f'nep_6p2arcsec_det2_ch{ch_name}.fits', overwrite=True)

    clear_output(wait=True)

Processing channels 3


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1834/1834 [00:00<00:00, 1318764.54it/s]

Loading reference frame from: /home/thomasli/spherex/selfcal/outputs/nep_det2_6p2arcsec/ref.fits



Building A, b matrix: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 1834/1834 [06:03<00:00,  5.04it/s]


Solving least squares for 63924716 unknowns with 218352247 equations.
 
LSQR            Least-squares solution of  Ax = b
The matrix A has 218352247 rows and 63924716 columns
damp = 1.00000000000000e-02   calc_var =        0
atol = 1.00e-06                 conlim = 1.00e+08
btol = 1.00e-06               iter_lim =      300
 
   Itn      x[0]       r1norm     r2norm   Compatible    LS      Norm A   Cond A
     0  0.00000e+00   2.718e+04  2.718e+04    1.0e+00  1.5e-01
     1  0.00000e+00   3.758e+03  3.758e+03    1.4e-01  8.4e-01   4.2e+03  1.0e+00
     2  0.00000e+00   1.887e+03  1.887e+03    6.9e-02  3.0e-01   5.9e+03  2.0e+00
     3  0.00000e+00   1.613e+03  1.613e+03    5.9e-02  1.6e-01   6.8e+03  3.1e+00
     4  0.00000e+00   1.506e+03  1.506e+03    5.5e-02  8.7e-02   7.7e+03  4.6e+00
     5  0.00000e+00   1.469e+03  1.469e+03    5.4e-02  5.7e-02   8.5e+03  6.0e+00
     6  0.00000e+00   1.443e+03  1.443e+03    5.3e-02  4.9e-02   9.1e+03  7.9e+00
     7  0.00000e+00   1.425e+03  1.42

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1834/1834 [00:00<00:00, 1375353.75it/s]

Loading reference frame from: /home/thomasli/spherex/selfcal/outputs/nep_det2_6p2arcsec/ref.fits


Calibration loaded from /home/thomasli/spherex/selfcal/outputs/nep_det2_6p2arcsec/calibration/cal_det2_ch3.h5


Sigma-clipped coadd:   0%|                                                                                                                       | 0/40 [00:00<?, ?it/s]

In [23]:
def plot_map(map, wcs):
    high, low = np.nanpercentile(map[map>0], [97, 0.001])
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection=wcs)
    im = ax.imshow(map, norm=LogNorm(vmin=low, vmax=high), origin='lower')

    # Explicitly set axis labels
    ax.coords['ra'].set_axislabel('RA')
    ax.coords['ra'].set_axislabel_position('b')  # Ensure RA label is only at the bottom
    ax.coords['ra'].set_ticks_position('b')  # Set RA ticks only at the bottom
    ax.coords['ra'].set_ticklabel_position('b')  # Set RA tick labels only at the bottom
    ax.coords['dec'].set_axislabel('DEC')

    # Add grid overlay
    ax.grid(color='black', linestyle='--', alpha=0.5)

    # Rescale the colorbar to match the height of the plot
    cbar = plt.colorbar(im, ax=ax, orientation='vertical', fraction=0.040, pad=0.04)
    cbar.set_label('MJy/sr')

In [4]:
# plot_map(sc, mm.ref_wcs)

In [5]:
# plt.imshow(mean-sc, vmin=-0.01, vmax=0.01, cmap='coolwarm', origin='lower')

In [73]:
reproj_file = mm.reproj_list[0]

In [97]:
importlib.reload(MakeMap)


<module 'MakeMap' from '/home/thomasli/spherex/selfcal/MakeMap.py'>

In [76]:
foo = MakeMap._prep_subframe(reproj_file, None, None, 0, 0, True, True, chunk_map, chunk_valid_mask)

In [77]:
file, exp_offset, det_offset, exp_idx, det_idx, apply_weight, apply_mask, chunk_map, chunk_valid_mask = (reproj_file, None, None, 0, 0, True, True, chunk_map, chunk_valid_mask)

In [ ]:
from MapHelper import bit_to_bool, make_weight, find_outliers, map_pixels, det_to_sub, compute_chunk_contrib, compute_crop, bin2d
from WCSHelper import load_from_fits, save_to_fits, find_optimal_frame, upscale_wcs

In [96]:
fields=['sub_data', 'ref_coords', 'grid_mapping']
if apply_mask:
    fields.append('grid_bitmask') # More like grid validity weight
result = MakeMap.load_reproj_file(file, fields=fields)


In [86]:
data = result['sub_data']
coords = result['ref_coords']
grid = result['grid_mapping']
# Ensure grid is not None before trying to access its shape
oversample_factor = int(grid.shape[-1] / data.shape[-1]) if grid is not None and data.shape[-1] > 0 else 1


In [87]:
# Apply mask
if apply_mask:
    bitmask = result['grid_bitmask']
    sub_mask = MakeMap.grid_bitmask_to_sub_mask(bitmask, oversample_factor, ignore_list=[17, 21], valid_threshold=0.99)
    data[~sub_mask] = np.nan

In [83]:
# Apply exposure offset if provided
if exp_offset is not None:
    data -= exp_offset[exp_idx]

# Compute chunk contribution to the sub-frame
chunk_contrib = None # Initialize chunk_contrib
if chunk_map is not None and grid is not None: # Check grid is not None
    chunk_contrib = compute_chunk_contrib(
        grid_mapping=grid,
        chunk_map=chunk_map,
        oversample_factor=oversample_factor
    )

# Apply detector offset ONLY IF det_offset AND chunk_contrib are available
if det_offset is not None and chunk_contrib is not None:
    sub_offset_flat = chunk_contrib.T @ det_offset[det_idx].flatten()
    sub_offset = sub_offset_flat.reshape(data.shape) # Use data.shape for consistency
    data -= sub_offset

# If chunk_valid_mask is provided, compute the contribution weight
chunk_weight = 1.0
if chunk_valid_mask is not None and chunk_contrib is not None:
    chunk_weight_flat = chunk_contrib.T @ chunk_valid_mask.flatten()
    chunk_weight = chunk_weight_flat.reshape(data.shape) # Use data.shape

weight = make_weight(data) if apply_weight else np.ones_like(data, dtype=np.float32)
weight *= chunk_weight # chunk_weight is 1.0 if not modified